# Retrieval & Reranking Evaluation
This notebook evaluates different retrieval and reranking strategies for plant disease information.

**Comparisons:**
1.  **Embeddings**: Google `gemini-embedding-001` vs OpenAI `text-embedding-3-large`
2.  **Sparse/Hybrid**: `SPLADE` vs `BM25`
3.  **Rerankers**: `Voyage AI rerank-2.5` vs `Jina AI jina-reranker-v3`

**Metrics:**
- NDCG@5
- Recall@5

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pathlib import Path
import uuid

# Vector DB & Embeddings
from qdrant_client import QdrantClient, models
from langchain_qdrant import FastEmbedSparse
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from fastembed import LateInteractionTextEmbedding

# Reranking
import voyageai
import requests

# Evaluation
from ranx import Qrels, Run, evaluate
from rank_bm25 import BM25Okapi

# Load environment variables
load_dotenv()

# Configuration
class Config:
    QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
    COLLECTION_NAME = "knowledgebase_collection"
    
    # API Keys
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
    JINA_API_KEY = os.getenv("JINA_API_KEY")
    
    # Models
    GEMINI_MODEL = "models/gemini-embedding-001"
    OPENAI_MODEL = "text-embedding-3-large"
    
    # Paths
    CORPUS_PATH = Path("data/corpus.parquet")
    QA_PATH = Path("data/qa.parquet")

# Initialize Clients
client = QdrantClient(url=Config.QDRANT_URL)


In [2]:
LateInteractionTextEmbedding.list_supported_models()

[{'model': 'colbert-ir/colbertv2.0',
  'sources': {'hf': 'colbert-ir/colbertv2.0',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'model.onnx',
  'description': 'Late interaction model',
  'license': 'mit',
  'size_in_GB': 0.44,
  'additional_files': [],
  'dim': 128,
  'tasks': {}},
 {'model': 'answerdotai/answerai-colbert-small-v1',
  'sources': {'hf': 'answerdotai/answerai-colbert-small-v1',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'vespa_colbert.onnx',
  'description': 'Text embeddings, Unimodal (text), Multilingual (~100 languages), 512 input tokens truncation, 2024 year',
  'license': 'apache-2.0',
  'size_in_GB': 0.13,
  'additional_files': [],
  'dim': 96,
  'tasks': {}},
 {'model': 'jinaai/jina-colbert-v2',
  'sources': {'hf': 'jinaai/jina-colbert-v2',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'onnx/model.onnx',
  'description': 'New model that expands capabilities of colbert-v1 with multilingual and 

In [2]:
# Load Data
if Config.CORPUS_PATH.exists() and Config.QA_PATH.exists():
    corpus_df = pd.read_parquet(Config.CORPUS_PATH)
    qa_df = pd.read_parquet(Config.QA_PATH)
    print(f"Loaded Corpus: {len(corpus_df)} documents")
    print(f"Loaded QA: {len(qa_df)} queries")
else:
    raise FileNotFoundError("Corpus or QA dataset not found.")

# Prepare Qrels (Ground Truth) for ranx
# qa_df['retrieval_gt'] contains list of list of doc_ids. 
# We need to convert it to a dictionary format for ranx: {qid: {doc_id: score}}
# Assuming binary relevance (1) for ground truth documents.

qrels_dict = {}
for _, row in qa_df.iterrows():
    qid = str(row['qid'])
    # retrieval_gt is often a list of lists in AutoRAG/some formats, or just list of strings.
    # Based on explore.ipynb output: [[17098978-5198-42ce-a808-a6c6b237f383]]
    # It seems to be a list of lists.
    
    gt_ids = row['retrieval_gt']
    qrels_dict[qid] = {}
    
    # Handle potential list of lists or simple list
    # Based on the sanity check, it seems we might have numpy arrays or lists wrapping the IDs
    for item in gt_ids:
        # If item is a list or numpy array, iterate through it
        if isinstance(item, (list, np.ndarray)):
            for doc_id in item:
                qrels_dict[qid][str(doc_id)] = 1
        else:
            # If it's a string or other scalar, use it directly
            qrels_dict[qid][str(item)] = 1

qrels = Qrels(qrels_dict)
print("Qrels prepared.")


Loaded Corpus: 2469 documents
Loaded QA: 600 queries
Qrels prepared.


## Sanity Check: Verify Qrels Setup
Before running the full evaluation, let's verify the qrels are properly formatted and check the ground truth data.

In [ ]:
# Sanity Check 1: Inspect Qrels Structure
print("=== QRELS SANITY CHECK ===")
print(f"Number of queries in qrels: {len(qrels_dict)}")
print(f"\nFirst 3 qrels entries:")
for i, (qid, docs) in enumerate(list(qrels_dict.items())[:3]):
    print(f"\nQuery ID: {qid}")
    print(f"  Ground truth docs: {docs}")
    
# Sanity Check 2: Inspect QA DataFrame
print("\n=== QA DATAFRAME CHECK ===")
print(f"QA DataFrame columns: {qa_df.columns.tolist()}")
print(f"\nFirst query example:")
first_row = qa_df.iloc[0]
print(f"  QID: {first_row['qid']}")
print(f"  Query: {first_row['query']}")
print(f"  Retrieval GT: {first_row['retrieval_gt']}")
print(f"  GT Type: {type(first_row['retrieval_gt'])}")

# Sanity Check 3: Verify corpus doc_ids format
print("\n=== CORPUS DOC_ID CHECK ===")
print(f"First 3 corpus doc_ids:")
for i in range(min(3, len(corpus_df))):
    print(f"  {corpus_df.iloc[i]['doc_id']} (type: {type(corpus_df.iloc[i]['doc_id'])})")

# Sanity Check 4: Check if ground truth IDs exist in corpus
print("\n=== ID MATCH CHECK ===")
corpus_ids_set = set(str(doc_id) for doc_id in corpus_df['doc_id'].tolist())
total_gt_docs = 0
matched_gt_docs = 0

for qid, docs in qrels_dict.items():
    for doc_id in docs.keys():
        total_gt_docs += 1
        if doc_id in corpus_ids_set:
            matched_gt_docs += 1

print(f"Total ground truth documents: {total_gt_docs}")
print(f"Matched in corpus: {matched_gt_docs}")
print(f"Match rate: {matched_gt_docs/total_gt_docs*100:.2f}%")

if matched_gt_docs == 0:
    print("\nWARNING: No ground truth IDs found in corpus!")
    print("This will cause 0 scores in evaluation.")
    print("\nChecking ID format mismatch...")
    sample_gt_id = list(list(qrels_dict.values())[0].keys())[0]
    sample_corpus_id = str(corpus_df.iloc[0]['doc_id'])
    print(f"Sample GT ID: '{sample_gt_id}' (type: {type(sample_gt_id)})")
    print(f"Sample Corpus ID: '{sample_corpus_id}' (type: {type(sample_corpus_id)})")

=== QRELS SANITY CHECK ===
Number of queries in qrels: 600

First 3 qrels entries:

Query ID: da25227f-1b4e-4907-81e2-a3b3fef62300
  Ground truth docs: {'dc9b290175ed088f3ca2693e3dbe6657': 1}

Query ID: bb8ccee4-864e-44f4-9ea6-598836b4a546
  Ground truth docs: {'86e3f1bdfc14b608cf542abfc4c71dd6': 1}

Query ID: ef6bcdcb-ccd6-430b-9471-8c00c3debf41
  Ground truth docs: {'4ac60b94f1274d11049d9c1890f68841': 1}

=== QA DATAFRAME CHECK ===
QA DataFrame columns: ['qid', 'query', 'retrieval_gt', 'generation_gt']

First query example:
  QID: da25227f-1b4e-4907-81e2-a3b3fef62300
  Query: What are the basic requirements for Alfalfa to grow best, including the ideal soil characteristics, pH range, and the attributes of its root system?
  Retrieval GT: [array(['dc9b290175ed088f3ca2693e3dbe6657'], dtype=object)]
  GT Type: <class 'numpy.ndarray'>

=== CORPUS DOC_ID CHECK ===
First 3 corpus doc_ids:
  dc9b290175ed088f3ca2693e3dbe6657 (type: <class 'str'>)
  86e3f1bdfc14b608cf542abfc4c71dd6 (type: <cl

In [18]:
# Check for empty contents in corpus_df
empty_contents_count = corpus_df['contents'].isna().sum() + (corpus_df['contents'] == "").sum()
print(f"Total documents: {len(corpus_df)}")
print(f"Documents with empty contents: {empty_contents_count}")

if empty_contents_count > 0:
    print("\nSample empty documents:")
    print(corpus_df[corpus_df['contents'] == ""].head())
else:
    print("No empty documents found in corpus_df.")

Total documents: 2469
Documents with empty contents: 0
No empty documents found in corpus_df.


## 1. Embedding Comparison: Gemini vs OpenAI
We will create two collections in Qdrant, one for each embedding model, and evaluate retrieval performance.


In [19]:
# Initialize Embedding Models
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model=Config.GEMINI_MODEL,
    google_api_key=Config.GOOGLE_API_KEY
)

openai_embeddings = OpenAIEmbeddings(
    model=Config.OPENAI_MODEL,
    openai_api_key=Config.OPENAI_API_KEY
)

# Initialize Sparse Embedding (SPLADE)
# Using Prithivida/Splade_PP_en_v1 as standard
splade_embeddings = FastEmbedSparse(model_name="prithivida/Splade_PP_en_v1")

print("Embedding models initialized.")


Embedding models initialized.


In [ ]:
def ingest_corpus_unified(collection_name, gemini_model, openai_model, splade_model):
    """Ingests corpus into a single Qdrant collection with multiple named vectors."""
    
    # Recreate collection
    client.delete_collection(collection_name=collection_name)
    
    # Define named vectors
    vectors_config = {
        "gemini": models.VectorParams(size=3072, distance=models.Distance.COSINE),
        "openai": models.VectorParams(size=3072, distance=models.Distance.COSINE)
    }
    
    # Define named sparse vectors
    sparse_vectors_config = {
        "splade": models.SparseVectorParams(),
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
        
    client.create_collection(
        collection_name=collection_name,
        vectors_config=vectors_config,
        sparse_vectors_config=sparse_vectors_config
    )

    # Create payload indexes for metadata fields to support efficient and case-insensitive filtering
    print("Creating payload indexes...")
    client.create_payload_index(collection_name=collection_name, field_name="metadata.plant", field_schema=models.PayloadSchemaType.TEXT)
    client.create_payload_index(collection_name=collection_name, field_name="metadata.disease", field_schema=models.PayloadSchemaType.TEXT)
    client.create_payload_index(collection_name=collection_name, field_name="metadata.type", field_schema=models.PayloadSchemaType.KEYWORD)
    client.create_payload_index(collection_name=collection_name, field_name="metadata.source", field_schema=models.PayloadSchemaType.KEYWORD)
    
    points = []
    batch_size = 25
    
    print(f"Ingesting into unified collection: {collection_name}...")
    docs = corpus_df.to_dict('records')
    
    for i, doc in enumerate(tqdm(docs)):
        content = doc['contents']
        doc_id = doc['doc_id']
        metadata = doc['metadata'] if doc['metadata'] else {}
        
        # Generate Dense & SPLADE
        gemini_vec = gemini_model.embed_query(content)
        openai_vec = openai_model.embed_query(content)
        splade_vec = splade_model.embed_query(content)
        
        vector_dict = {
            "gemini": gemini_vec,
            "openai": openai_vec,
            "splade": models.SparseVector(
                indices=splade_vec.indices,
                values=splade_vec.values
            ),
            # Use Qdrant's BM25 model
            "bm25": models.Document(
                text=content,
                model="Qdrant/bm25"
            )
        }
            
        point = models.PointStruct(
            id=doc_id,
            vector=vector_dict,
            payload={
                "page_content": content,
                "metadata": metadata
            }
        )
        points.append(point)
        
        if len(points) >= batch_size:
            client.upsert(collection_name=collection_name, points=points)
            points = []
            
    if points:
        client.upsert(collection_name=collection_name, points=points)
        
    print(f"Unified ingestion complete.")

# Run unified ingestion
ingest_corpus_unified(
    Config.COLLECTION_NAME, 
    gemini_embeddings, 
    openai_embeddings, 
    splade_embeddings
)

In [21]:
client.get_collection(collection_name=Config.COLLECTION_NAME)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=4916, points_count=2458, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors={'gemini': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'openai': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors={'bm25': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>), 'splade': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, 

In [22]:
def evaluate_retrieval_unified(collection_name, query_text, qid, mode="gemini", k=20):
    """Performs retrieval using named vectors in the unified collection."""
    
    if mode == "gemini":
        vec = gemini_embeddings.embed_query(query_text)
        results = client.query_points(collection_name=collection_name, query=vec, using="gemini", limit=k, with_payload=True)
    elif mode == "openai":
        vec = openai_embeddings.embed_query(query_text)
        results = client.query_points(collection_name=collection_name, query=vec, using="openai", limit=k, with_payload=True)
    elif mode == "splade":
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
            using="splade",
            limit=k,
            with_payload=True
        )
    elif mode == "bm25":
        results = client.query_points(
            collection_name=collection_name,
            query=models.Document(text=query_text, model="Qdrant/bm25"),
            using="bm25",
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_gemini_splade":
        dense_vec = gemini_embeddings.embed_query(query_text)
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="gemini", limit=k),
                models.Prefetch(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_openai_splade":
        dense_vec = openai_embeddings.embed_query(query_text)
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="openai", limit=k),
                models.Prefetch(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_gemini_bm25":
        dense_vec = gemini_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="gemini", limit=k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_openai_bm25":
        dense_vec = openai_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="openai", limit=k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    
    return results

def evaluate_retrieval_batch_unified(collection_name, queries, mode="gemini", k=20, batch_size=32):
    """Performs batch retrieval using named vectors in the unified collection."""
    all_results = []
    
    for i in tqdm(range(0, len(queries), batch_size), desc=f"Batch Retrieval ({mode})"):
        batch_queries = queries[i : i + batch_size]
        query_texts = [q['query'] for q in batch_queries]
        
        requests = []
        
        if mode == "gemini":
            vecs = gemini_embeddings.embed_documents(query_texts)
            for vec in vecs:
                requests.append(models.QueryRequest(query=vec, using="gemini", limit=k, with_payload=True))
        elif mode == "openai":
            vecs = openai_embeddings.embed_documents(query_texts)
            for vec in vecs:
                requests.append(models.QueryRequest(query=vec, using="openai", limit=k, with_payload=True))
        elif mode == "splade":
            sparse_vecs = splade_embeddings.embed_documents(query_texts)
            for sparse_vec in sparse_vecs:
                requests.append(models.QueryRequest(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k,
                    with_payload=True
                ))
        elif mode == "bm25":
            for text in query_texts:
                requests.append(models.QueryRequest(
                    query=models.Document(text=text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k,
                    with_payload=True
                ))
        elif mode.startswith("hybrid"):
            # Handle hybrid modes
            is_gemini = "gemini" in mode
            is_splade = "splade" in mode
            
            dense_model = gemini_embeddings if is_gemini else openai_embeddings
            dense_using = "gemini" if is_gemini else "openai"
            
            dense_vecs = dense_model.embed_documents(query_texts)
            
            if is_splade:
                sparse_vecs = splade_embeddings.embed_documents(query_texts)
                for dv, sv in zip(dense_vecs, sparse_vecs):
                    requests.append(models.QueryRequest(
                        prefetch=[
                            models.Prefetch(query=dv, using=dense_using, limit=k),
                            models.Prefetch(
                                query=models.SparseVector(indices=sv.indices, values=sv.values),
                                using="splade",
                                limit=k
                            ),
                        ],
                        query=models.FusionQuery(fusion=models.Fusion.RRF),
                        limit=k,
                        with_payload=True
                    ))
            else: # bm25
                for dv, text in zip(dense_vecs, query_texts):
                    requests.append(models.QueryRequest(
                        prefetch=[
                            models.Prefetch(query=dv, using=dense_using, limit=k),
                            models.Prefetch(
                                query=models.Document(text=text, model="Qdrant/bm25"),
                                using="bm25",
                                limit=k
                            ),
                        ],
                        query=models.FusionQuery(fusion=models.Fusion.RRF),
                        limit=k,
                        with_payload=True
                    ))
        
        batch_res = client.query_batch_points(collection_name=collection_name, requests=requests)
        all_results.extend(batch_res)
        
    return all_results

In [23]:
import pickle
from pathlib import Path

RESULTS_FILE = Path("data/evaluation_results.pkl")

def save_evaluation_state():
    # Convert Run objects to dicts to avoid pickling issues with Numba objects
    serializable_runs = {name: run.to_dict() for name, run in unified_runs.items()}
    state = {
        "unified_results": unified_results,
        "unified_runs": serializable_runs
    }
    # Ensure directory exists
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_FILE, "wb") as f:
        pickle.dump(state, f)
    print(f"Evaluation state saved to {RESULTS_FILE}")

def load_evaluation_state():
    if RESULTS_FILE.exists():
        try:
            with open(RESULTS_FILE, "rb") as f:
                state = pickle.load(f)
            print(f"Loaded evaluation state from {RESULTS_FILE}")
            
            # Reconstruct Run objects
            loaded_results = state.get("unified_results", {})
            loaded_runs_dicts = state.get("unified_runs", {})
            loaded_runs = {name: Run.from_dict(d, name=name) for name, d in loaded_runs_dicts.items()}
            
            return loaded_results, loaded_runs
        except Exception as e:
            print(f"Failed to load evaluation state: {e}")
            return {}, {}
    return {}, {}

if 'unified_results' not in globals() or not unified_results:
    unified_results, unified_runs = load_evaluation_state()

# Ensure variables exist even if load failed or file didn't exist
if 'unified_results' not in globals():
    unified_results = {}
if 'unified_runs' not in globals():
    unified_runs = {}

In [24]:
def run_single_evaluation(mode, collection_name=Config.COLLECTION_NAME, k=5, sample_n=None, overwrite=False):
    """Runs evaluation for a single mode. Use sample_n=50 for a quick sanity check."""
    
    # Check if already evaluated and not forcing overwrite
    if not overwrite and not sample_n and mode in unified_results:
        print(f"Mode '{mode}' already evaluated. Loading results...")
        return unified_results[mode]

    # 1. Select Queries (Full or Sample)
    if sample_n:
        # Randomly sample n queries with fixed seed for reproducibility
        queries_df = qa_df.sample(n=min(sample_n, len(qa_df)), random_state=42)
        print(f"⚠️ DEBUG MODE: Running on {len(queries_df)} random queries.")
    else:
        queries_df = qa_df
        print(f"Evaluating mode: {mode} with k={k} on full dataset...")

    queries = queries_df.to_dict('records')
    run_dict = {}
    
    # 2. Run Retrieval (Batched)
    results = evaluate_retrieval_batch_unified(collection_name, queries, mode=mode, k=k)
    
    for q, res in zip(queries, results):
        qid = str(q['qid'])
        run_dict[qid] = {}
        max_rank = k
        for point in res.points:
            str_id = str(point.id).replace("-", "")
            run_dict[qid][str_id] = max_rank
            max_rank -= 1
    
    run = Run(run_dict, name=mode)
    
    # 3. Handle Qrels (Ground Truth)
    # If we only run 50 queries, we must evaluate against ONLY those 50 qrels 
    # otherwise ranx assumes we missed the other 950 queries and gives ~0 score.
    if sample_n:
        sampled_qids = set(run_dict.keys())
        # Filter the global qrels_dict to match our sample
        sampled_qrels_dict = {qid: val for qid, val in qrels_dict.items() if qid in sampled_qids}
        eval_qrels = Qrels(sampled_qrels_dict)
    else:
        eval_qrels = qrels

    metrics = evaluate(eval_qrels, run, metrics=["ndcg@5", "recall@5", "mrr@5"], make_comparable=True)
    
    # 4. Store Results (Only for full runs)
    if not sample_n:
        unified_results[mode] = metrics
        unified_runs[mode] = run
        save_evaluation_state()
    else:
        print(f"⚠️ Results are for SAMPLE only and NOT saved to global stats.")
    
    print(f"Results for {mode}: {metrics}")
    return metrics

## Dense Comparisons

In [25]:
run_single_evaluation("gemini")
run_single_evaluation("openai")


# Map available results to variables (use .get() to avoid errors if not run yet)
gemini_results = unified_results.get('gemini')
openai_results = unified_results.get('openai')


Mode 'gemini' already evaluated. Loading results...
Mode 'openai' already evaluated. Loading results...


## Hybrid Comparison


In [26]:
run_single_evaluation("hybrid_gemini_splade")
run_single_evaluation("hybrid_openai_splade")
run_single_evaluation("hybrid_gemini_bm25")
run_single_evaluation("hybrid_openai_bm25")

gemini_hybrid_results = unified_results.get('hybrid_gemini_splade')
openai_hybrid_results = unified_results.get('hybrid_openai_splade')
gemini_bm25_results = unified_results.get('hybrid_gemini_bm25')
openai_bm25_results = unified_results.get('hybrid_openai_bm25')

Mode 'hybrid_gemini_splade' already evaluated. Loading results...
Mode 'hybrid_openai_splade' already evaluated. Loading results...
Mode 'hybrid_gemini_bm25' already evaluated. Loading results...
Mode 'hybrid_openai_bm25' already evaluated. Loading results...


## 3. Reranker Comparison: Voyage AI vs Jina AI



In [27]:
# Sanity Check: Test VoyageAIRerank
try:
    from langchain_voyageai import VoyageAIRerank
    from langchain_core.documents import Document as LCDocument

    print("Testing VoyageAIRerank initialization...")
    test_reranker = VoyageAIRerank(
        model="rerank-2.5",
        voyageai_api_key=Config.VOYAGE_API_KEY,
        top_k=2
    )

    test_docs = [
        LCDocument(page_content="Apple is a fruit.", metadata={"id": "1"}),
        LCDocument(page_content="The capital of France is Paris.", metadata={"id": "2"}),
        LCDocument(page_content="Python is a programming language.", metadata={"id": "3"})
    ]
    test_query = "What is the capital of France?"

    print("Testing reranking...")
    test_results = test_reranker.compress_documents(test_docs, test_query)
    
    print("\nReranking Results:")
    for i, doc in enumerate(test_results):
        print(f"{i+1}. {doc.page_content} (ID: {doc.metadata.get('id')})")
        
except Exception as e:
    print(f"Sanity Check Failed: {e}")
    import traceback
    traceback.print_exc()

Testing VoyageAIRerank initialization...
Testing reranking...

Reranking Results:
1. The capital of France is Paris. (ID: 2)
2. Apple is a fruit. (ID: 1)


In [28]:
def get_candidates(mode="gemini", collection_name=Config.COLLECTION_NAME, k=100):
    """Retrieves top-k candidates for all queries using the specified retrieval mode (Batched)."""
    candidates = {}
    queries = qa_df.to_dict('records')
    
    print(f"Fetching candidates with mode: {mode} (k={k})...")
    
    # Use the batch retrieval function
    results = evaluate_retrieval_batch_unified(collection_name, queries, mode=mode, k=k)
    
    for q, res in zip(queries, results):
        qid = str(q['qid'])
        query_text = q['query']
        
        # Store candidates as list of (doc_id, content)
        doc_list = []
        for point in res.points:
            # Try multiple common keys for the content to be robust
            payload = point.payload or {}
            content = payload.get('page_content') or payload.get('contents') or payload.get('text') or ""
            
            doc_id = str(point.id).replace("-", "")
            doc_list.append((doc_id, content))
            
        candidates[qid] = {
            "query": query_text,
            "docs": doc_list
        }
    return candidates

# === CONFIGURATION ===
# Choose the base model to retrieve candidates for reranking
BASE_RETRIEVAL_MODE = "hybrid_gemini_bm25" 
CANDIDATE_K = 50 

def evaluate_reranker(reranker_name, candidates, overwrite=False):
    key = f"{reranker_name}_rerank"
    
    if not overwrite and key in unified_results:
        print(f"Reranker '{reranker_name}' already evaluated. Loading results...")
        return unified_results[key], unified_runs[key]

    run_dict = {}
    print(f"Reranking with {reranker_name}...")
    
    # Initialize rerankers outside the loop
    if reranker_name == "voyage":
        from langchain_voyageai import VoyageAIRerank
        reranker = VoyageAIRerank(
            model="rerank-2.5",
            voyageai_api_key=Config.VOYAGE_API_KEY,
            top_k=CANDIDATE_K
        )
        from langchain_core.documents import Document as LCDocument

    for qid, data in tqdm(candidates.items()):
        query = data['query']
        docs = data['docs'] # list of (id, content)
        
        if not docs:
            continue
            
        if reranker_name == "voyage":
            try:
                # Filter out empty documents before reranking
                valid_docs = [(doc_id, content) for doc_id, content in docs if content and content.strip()]
                
                if not valid_docs:
                    continue
                
                # Prepare documents for reranking
                rerank_docs = [
                    LCDocument(page_content=content, metadata={"doc_id": doc_id})
                    for doc_id, content in valid_docs
                ]

                # Perform reranking
                reranked_results = reranker.compress_documents(rerank_docs, query)
                
                if not reranked_results:
                    continue

                # Assign rank-based scores
                max_rank = len(reranked_results)
                run_dict[qid] = {}
                for res in reranked_results:
                    doc_id = res.metadata["doc_id"]
                    run_dict[qid][doc_id] = max_rank
                    max_rank -= 1
            except Exception as e:
                # Print only the first few errors to avoid flooding
                if len(run_dict) < 5:
                    print(f"Voyage Error for query '{query[:50]}...': {e}")
                
        elif reranker_name == "jina":
            doc_texts = [d[1] for d in docs]
            doc_ids = [d[0] for d in docs]
            
            # Hot Debugging Prints
            if len(run_dict) < 2:  # Print debug info for first 2 queries
                print(f"\n--- Jina Debug (Query {qid}) ---")
                print(f"Query: {query[:100]}...")
                print(f"Num Docs: {len(doc_texts)}")
                print(f"First Doc Content (sample): {doc_texts[0][:100] if doc_texts else 'EMPTY'}...")
                print(f"Config API Key: {Config.JINA_API_KEY[:5]}***" if Config.JINA_API_KEY else "MISSING")

            # Jina AI Rerank
            url = "https://api.jina.ai/v1/rerank"
            headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {Config.JINA_API_KEY}"
            }
            payload = {
                "model": "jina-reranker-v3",
                "query": query,
                "documents": doc_texts,
                "top_n": len(doc_texts)
            }
            try:
                response = requests.post(url, headers=headers, json=payload)
                resp = response.json()
                
                if len(run_dict) < 2:
                    print(f"Jina Status Code: {response.status_code}")
                    if 'results' not in resp:
                        print(f"Jina Full Response: {resp}")

                if 'results' in resp:
                    max_rank = len(resp['results'])
                    run_dict[qid] = {}
                    for res in resp['results']:
                        idx = res['index']
                        doc_id = doc_ids[idx]
                        run_dict[qid][doc_id] = max_rank
                        max_rank -= 1
                else:
                    if len(run_dict) < 5:
                        print(f"Jina Error Response: {resp}")
            except Exception as e:
                if len(run_dict) < 5:
                    print(f"Jina Error: {e}")

    if not run_dict:
        print(f"⚠️ No results collected for {reranker_name}. Check for API errors above.")
        return None, None

    run = Run(run_dict, name=f"{reranker_name}_on_{BASE_RETRIEVAL_MODE}")
    # Use make_comparable=True to handle cases where some queries failed to return results
    metrics = evaluate(qrels, run, metrics=["ndcg@5", "recall@5", "mrr@5"], make_comparable=True)
    
    # Save Results
    unified_results[key] = metrics
    unified_runs[key] = run
    save_evaluation_state()
    
    return metrics, run

# Check if we need to generate candidates (only if rerankers are NOT already evaluated)
rerankers_to_run = []
if 'voyage_rerank' not in unified_results: rerankers_to_run.append("voyage")
if 'jina_rerank' not in unified_results: rerankers_to_run.append("jina")

# FORCE generate candidates for debugging if we are looking at Jina issues
candidates = get_candidates(mode=BASE_RETRIEVAL_MODE, k=CANDIDATE_K)

# Evaluate Voyage
voyage_results, voyage_run = evaluate_reranker("voyage", candidates)
print("Voyage Results:", voyage_results)

# Evaluate Jina (FORCED OVERWRITE FOR DEBUGGING)
jina_results, jina_run = evaluate_reranker("jina", candidates, overwrite=True)
print("Jina Results:", jina_results)

Fetching candidates with mode: hybrid_gemini_bm25 (k=50)...


Batch Retrieval (hybrid_gemini_bm25):   0%|          | 0/19 [00:00<?, ?it/s]

Reranker 'voyage' already evaluated. Loading results...
Voyage Results: {'ndcg@5': 0.956029935791571, 'recall@5': 0.98, 'mrr@5': 0.9581666666666668}
Reranking with jina...


  0%|          | 0/600 [00:00<?, ?it/s]


--- Jina Debug (Query da25227f-1b4e-4907-81e2-a3b3fef62300) ---
Query: What are the basic requirements for Alfalfa to grow best, including the ideal soil characteristics, ...
Num Docs: 50
First Doc Content (sample): Cultivation and propagation of Alfalfa (Part 1)

Basic Requirements Alfalfa is adapted to grow in a ...
Config API Key: jina_***
Jina Status Code: 403
Jina Full Response: {'detail': 'Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.', 'request_id': '0438693297583cddfd546ca8f342fe82', 'code': 'AUTHZ_INSUFFICIENT_BALANCE'}
Jina Error Response: {'detail': 'Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.', 'request_id': '0438693297583cddfd546ca8f342fe82', 'code': 'AUTHZ_INSUFFICIENT_BALANCE'}

--- Jina Debug (Query bb8ccee4-864e-44f4-9ea6-598836b4a546) ---
Query: What guideline dictates that you can plant alfalfa seed shallower if there's enough moisture, but sh...
Num Docs: 50
First 

KeyboardInterrupt: 

## Summary of Results


In [32]:
# Retrieve results from unified_results dictionary
results_summary = {
    "Gemini (Dense)": unified_results.get("gemini"),
    "OpenAI (Dense)": unified_results.get("openai"),
    "Gemini + SPLADE": unified_results.get("hybrid_gemini_splade"),
    "OpenAI + SPLADE": unified_results.get("hybrid_openai_splade"),
    "Gemini + BM25": unified_results.get("hybrid_gemini_bm25"),
    "OpenAI + BM25": unified_results.get("hybrid_openai_bm25"),
    "Gemini + BM25 + Voyage Rerank": unified_results.get("voyage_rerank"),
}

# Filter out None values (models not yet run)
results_summary = {k: v for k, v in results_summary.items() if v is not None}

df_results = pd.DataFrame(results_summary).T
df_results.sort_values(by="ndcg@5", ascending=False, inplace=True)
df_results

,ndcg@5,recall@5,mrr@5
Gemini + BM25 + Voyage Rerank,0.956030,0.980000,0.958167
Gemini (Dense),0.939847,0.975833,0.936611
Gemini + BM25,0.939335,0.968333,0.944917
OpenAI + BM25,0.936182,0.967500,0.941861
Gemini + SPLADE,0.930851,0.965833,0.931306
OpenAI + SPLADE,0.927047,0.958333,0.934139
OpenAI (Dense),0.925189,0.957500,0.931556


## Statistical Significance & Visualization
We compare the top performing models to see if the differences are statistically significant and visualize their performance across different recall levels.

In [34]:
from ranx import compare

# Compare Dense Embeddings
print("=== Statistical Comparison: Gemini vs OpenAI ===")
if 'gemini' in unified_runs and 'openai' in unified_runs:
    report = compare(
        qrels=qrels,
        runs=[unified_runs['gemini'], unified_runs['openai']],
        metrics=["ndcg@5", "mrr@5", "recall@5"],
        max_p=0.05,
        make_comparable=True
    )
    print(report)
else:
    print("Skipping comparison: Missing gemini or openai runs.")

# Compare Best Retrieval vs Rerankers
print("\n=== Statistical Comparison: Hybrid vs Rerankers ===")
rerank_runs = []
if 'hybrid_gemini_bm25' in unified_runs: rerank_runs.append(unified_runs['hybrid_gemini_bm25'])
if 'voyage_rerank' in unified_runs: rerank_runs.append(unified_runs['voyage_rerank'])

if len(rerank_runs) > 1:
    report_rerank = compare(
        qrels=qrels,
        runs=rerank_runs,
        metrics=["ndcg@5", "mrr@5"],
        max_p=0.05,
        make_comparable=True
    )
    print(report_rerank)
else:
    print("Skipping comparison: Not enough runs.")

=== Statistical Comparison: Gemini vs OpenAI ===
#    Model    NDCG@5      MRR@5  Recall@5
---  -------  --------  -------  ----------
a    gemini   0.940ᵇ      0.937  0.976ᵇ
b    openai   0.925       0.932  0.958

=== Statistical Comparison: Hybrid vs Rerankers ===
#    Model               NDCG@5    MRR@5
---  ------------------  --------  -------
a    hybrid_gemini_bm25  0.939     0.945
b    voyage_rerank       0.956ᵃ    0.958ᵃ
